In [1]:
import os
import pandas as pd
import zipfile

TEMP_DIR = "../data/processed/temp"
MAIN_CSV = "../data/processed/dataset_final_qwen_filled.csv"
IMAGE_DIR = "../data/raw/images"
OUTPUT_ZIP = "../data/processed/remaining_images.zip"
REMAINING_CSV = "../data/processed/dataset_remaining.csv"

PROCESSED_FILES = [
    "dataset_qwen_filled_0_25.csv",
    "dataset_qwen_filled_25_50.csv",
    "dataset_qwen_filled_75_100.csv"
]

In [2]:
processed_ids = set()

for file_name in PROCESSED_FILES:
    file_path = os.path.join(TEMP_DIR, file_name)
    if os.path.exists(file_path):
        try:
            df_temp = pd.read_csv(file_path, dtype={'article_id': str})
            processed_ids.update(df_temp['article_id'].str.zfill(10))
            print(f"{file_name} loaded")
        except Exception as e:
            print(f"Error reading {file_name}: {e}")

print(f"total processed ids: {len(processed_ids)}")

dataset_qwen_filled_0_25.csv loaded
dataset_qwen_filled_25_50.csv loaded
dataset_qwen_filled_75_100.csv loaded
total processed ids: 41500


In [3]:
df_main = pd.read_csv(MAIN_CSV, dtype={'article_id': str})
df_main['article_id_str'] = df_main['article_id'].str.zfill(10)

df_remain = df_main[~df_main['article_id_str'].isin(processed_ids)].copy()
df_remain.drop(columns=['article_id_str'], inplace=True)
df_remain.to_csv(REMAINING_CSV, index=False)

total_remaining = len(df_remain)
print(f"remaining ids: {total_remaining}")
print("remaining csv saved")

remaining ids: 28697
remaining csv saved


In [4]:
found_count = 0
missing_count = 0

with zipfile.ZipFile(OUTPUT_ZIP, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for aid in df_remain['article_id'].str.zfill(10):
        folder = aid[:3]
        img_path = os.path.join(IMAGE_DIR, folder, f"{aid}.jpg")
        
        if os.path.exists(img_path):
            zipf.write(img_path, arcname=f"images/{folder}/{aid}.jpg")
            found_count += 1
        else:
            missing_count += 1

print(f"images zipped: {found_count}")
print(f"images missing: {missing_count}")

images zipped: 28697
images missing: 0


In [1]:
import os
import glob
import pandas as pd

TEMP_DIR = "../data/processed/temp"
OUTPUT_FILE = "../data/processed/dataset_qwen_completed.csv"

def merge_and_clean_csvs(temp_dir, output_file):
    all_files = glob.glob(os.path.join(temp_dir, "*.csv"))
    if not all_files:
        print("Không tìm thấy file CSV nào.")
        return
    
    print(f"Tìm thấy {len(all_files)} files. Đang tiến hành gộp và làm sạch...")
    
    df_list = []
    for file in all_files:
        # Ép kiểu string để tránh việc pandas tự động xóa mất số 0 ở đầu article_id (VD: 0599...)
        df = pd.read_csv(file, dtype=str) 
        
        # Danh sách các cột bị duplicate theo báo cáo
        target_cols = ['fit', 'occasion', 'seasonality']
        
        for col in target_cols:
            dup_col = f"{col}.1"
            if dup_col in df.columns:
                # Đưa các dạng rỗng về pandas NA để có thể dùng combine_first
                df[col] = df[col].replace(['', ' ', 'nan', 'NaN', 'Unknown'], pd.NA)
                df[dup_col] = df[dup_col].replace(['', ' ', 'nan', 'NaN', 'Unknown'], pd.NA)
                
                # Combine dữ liệu: Lấy ưu tiên từ cột gốc, nếu trống thì lấy từ cột .1
                df[col] = df[col].combine_first(df[dup_col])
                
                # Sau khi gộp, fill lại Unknown cho các ô thực sự không có data và xóa cột .1
                df[col] = df[col].fillna("Unknown")
                df = df.drop(columns=[dup_col])
        
        df_list.append(df)
        print(f"- Đã xử lý: {os.path.basename(file)} ({len(df)} dòng)")
        
    # Gộp tất cả dataframe thành 1
    final_df = pd.concat(df_list, ignore_index=True)
    
    # Loại bỏ các dòng trùng lặp ID (Trường hợp các file chunk bị chồng chéo ID trong lúc chạy)
    before_dedup = len(final_df)
    final_df = final_df.drop_duplicates(subset=['article_id'], keep='last')
    after_dedup = len(final_df)
    
    # Lưu ra file cuối cùng
    final_df.to_csv(output_file, index=False)
    
    print(f"\nHoàn tất! Đã lưu file tại: {output_file}")
    print(f"Tổng số bản ghi ban đầu: {before_dedup}")
    print(f"Tổng số bản ghi sau khi lọc ID trùng: {after_dedup}")

if __name__ == "__main__":
    merge_and_clean_csvs(TEMP_DIR, OUTPUT_FILE)

Tìm thấy 4 files. Đang tiến hành gộp và làm sạch...


C:\Users\tlkur\AppData\Local\Temp\ipykernel_4828\2248625610.py:29: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[dup_col] = df[dup_col].replace(['', ' ', 'nan', 'NaN', 'Unknown'], pd.NA)
C:\Users\tlkur\AppData\Local\Temp\ipykernel_4828\2248625610.py:29: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[dup_col] = df[dup_col].replace(['', ' ', 'nan', 'NaN', 'Unknown'], pd.NA)
C:\Users\tlkur\AppData\Local\Temp\ipykernel_4828\2248625610.py:29: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a f

- Đã xử lý: dataset_qwen_completed.csv (28697 dòng)
- Đã xử lý: dataset_qwen_filled_0_25.csv (13100 dòng)


C:\Users\tlkur\AppData\Local\Temp\ipykernel_4828\2248625610.py:29: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[dup_col] = df[dup_col].replace(['', ' ', 'nan', 'NaN', 'Unknown'], pd.NA)
C:\Users\tlkur\AppData\Local\Temp\ipykernel_4828\2248625610.py:29: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[dup_col] = df[dup_col].replace(['', ' ', 'nan', 'NaN', 'Unknown'], pd.NA)
C:\Users\tlkur\AppData\Local\Temp\ipykernel_4828\2248625610.py:29: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a f

- Đã xử lý: dataset_qwen_filled_25_50.csv (14300 dòng)
- Đã xử lý: dataset_qwen_filled_75_100.csv (14100 dòng)

Hoàn tất! Đã lưu file tại: ../data/processed/dataset_qwen_completed.csv
Tổng số bản ghi ban đầu: 70197
Tổng số bản ghi sau khi lọc ID trùng: 70197
